In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Synthetic Data

The classes of that section can train a model as soon as they
are given data. Which data should the first run use? Real datasets are what
we ultimately care about, but they make a poor first test. Suppose that we
fit a linear model to house prices like those of
that section and its predictions are poor. Three
explanations compete: the model may be
misspecified, the optimization algorithm or its implementation may be
flawed, or the data may be pathological. On real data these explanations
are difficult to tell apart.
*Synthetic data* removes the ambiguity by construction. If we generate the
data ourselves, we know the true weights $\mathbf{w}^*$, the true bias
$b^*$, and the distribution of the noise. A model of the right form then has
a known target, and any systematic failure to recover $\mathbf{w}^*$ and
$b^*$, beyond what the noise explains, points to the algorithm or its
implementation. A synthetic problem with known parameters, generated by a
model of the form that the method fits, is therefore a useful first test of
a new learning method, before we evaluate it on real data.

This section builds such a test for linear regression. We generate examples
from a linear function with the true parameters $\mathbf{w}^* = (2, -3.4)$
and $b^* = 4.2$, add a little Gaussian noise, and package the result as a
`DataModule`, the class for datasets from that section. We then
work out how close a correct fit can come to these parameters, which turns
the comparison into a test with a threshold. Finally, we write the data
loader that serves the examples in random minibatches, the form in which
minibatch stochastic gradient descent consumes them. The next section trains
a linear model on this dataset and holds its estimates to the threshold.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import random

## Generating the Data

We draw the features of each example independently from a standard normal
distribution, with mean $0$ and variance $1$, and stack the feature vectors
as the rows of the design matrix $\mathbf{X}$. Two features keep the true
parameters easy to compare; nothing below depends on that choice. Each label
is the value of the true linear function plus noise:

$$\mathbf{y} = \mathbf{X}\mathbf{w}^* + b^* + \boldsymbol{\epsilon}.$$

The bias $b^*$ is added to every entry, and the entries of the noise vector
$\boldsymbol{\epsilon}$ are drawn independently from a normal distribution
with mean $0$ and standard deviation $\sigma = 0.01$. Each label thus lies
close to the plane $\mathbf{w}^{*\top}\mathbf{x} + b^*$, typically within a
few hundredths. This is the model of that section: a
linear function with Gaussian noise, for which minimizing the squared loss
is the same as maximizing the likelihood
(that section). A linear model
trained with the squared loss is therefore correctly specified for these
data.

We write the generator as the constructor of `SyntheticRegressionData`, a
subclass of `DataModule`, which serves a dataset's training and validation
examples through the methods `train_dataloader` and `val_dataloader`. The
call `save_hyperparameters()` stores every constructor argument as an
attribute: the true parameters `w` and `b`, the noise level `noise`, the
numbers `num_train` and `num_val` of training and validation examples, and
the minibatch size `batch_size`. The constructor draws all
`num_train + num_val` examples at once, by default 1,000 of each. The
validation examples come from the same process but are not used to fit the
model. The loss on them shows how well the model predicts examples it has
not seen, a question we take up in that section.

JAX has no global random state. The constructor therefore takes a key,
splits it into two, and draws $\mathbf{X}$ and $\boldsymbol{\epsilon}$ from
different keys. The same key always produces the same dataset. Without a
key, the constructor draws one from a new seed, so that each run generates
a new dataset.

In [2]:
class SyntheticRegressionData(d2l.DataModule):
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32, key=None):
        super().__init__()
        self.save_hyperparameters()
        if key is None:  # A new seed on each run
            key = jax.random.key(random.getrandbits(32))
        n = num_train + num_val
        key1, key2 = jax.random.split(key)
        self.X = jax.random.normal(key1, (n, w.shape[0]))
        eps = jax.random.normal(key2, (n, 1)) * noise
        self.y = d2l.matmul(self.X, d2l.reshape(w, (-1, 1))) + b + eps

We set the true parameters to $\mathbf{w}^* = (2, -3.4)$ and $b^* = 4.2$,
the values that the next two sections train on as well.

In [3]:
data = SyntheticRegressionData(w=d2l.tensor([2, -3.4]), b=4.2)

Each row of `data.X` holds the two features of one example, and each row of
`data.y` its label. For the first example we also compute the noise, the
label minus the noise-free value $\mathbf{w}^{*\top}\mathbf{x} + b^*$.

In [4]:
x, y = data.X[0], data.y[0, 0]
noise = y - (d2l.reduce_sum(data.w * x) + data.b)
print(f'features: ({float(x[0]):.4f}, {float(x[1]):.4f})')
print(f'label: {float(y):.4f}, noise: {float(noise):.4f}')

features: (0.3637, -0.1650)
label: 5.4902, noise: 0.0017


The label differs from the value of the linear function only by the noise,
which is at most a few hundredths.

## Recovering the Parameters

A test needs a threshold. Even a perfect implementation does not recover
$\mathbf{w}^*$ and $b^*$ exactly, because the noise moves the labels and,
with them, the minimizer of the loss. How far the minimizer moves follows
from the normal equations of
that section. As there, we append a
column of ones to $\mathbf{X}$ and the bias to the weights, so that
$\mathbf{w}^* = (2, -3.4, 4.2)$ now has three entries and
$\mathbf{y} = \mathbf{X}\mathbf{w}^* + \boldsymbol{\epsilon}$. The
least-squares estimate $\hat{\mathbf{w}}$ on the $n$ training examples
solves the normal equations
$\mathbf{X}^\top\mathbf{X}\hat{\mathbf{w}} = \mathbf{X}^\top\mathbf{y}$.
Substituting the labels gives
$\mathbf{X}^\top\mathbf{X}(\hat{\mathbf{w}} - \mathbf{w}^*) = \mathbf{X}^\top\boldsymbol{\epsilon}$,
and therefore

$$\hat{\mathbf{w}} - \mathbf{w}^* = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\boldsymbol{\epsilon}.$$

The *estimation error* $\hat{\mathbf{w}} - \mathbf{w}^*$ is a linear
function of the noise, so without noise least squares would recover the
parameters exactly.

For our data the matrix $\mathbf{X}^\top\mathbf{X}$ is simple. Its entries
are sums over the $n$ examples of products of two columns. On the diagonal
each product is either the square of the constant $1$ or a squared feature
$z^2$, with mean $1$ and variance $\mathbb{E}[z^4] - 1 = 3 - 1 = 2$ for a standard
normal $z$. These sums are therefore close to $n$, within about
$\sqrt{2n}$. Off the diagonal the products have mean $0$, because the
features are independent with mean $0$, and variance $1$, so a sum of $n$ of
them has standard deviation only $\sqrt{n}$. For $n = 1{,}000$ the diagonal
entries are $1{,}000$ for the column of ones and about $1{,}000 \pm 45$ for
the features, and the entries off the diagonal are about $0 \pm 32$. With
$\mathbf{X}^\top\mathbf{X} \approx n\mathbf{I}$, the estimation error
becomes an average over the training examples,

$$\hat{\mathbf{w}} - \mathbf{w}^* \approx \frac{1}{n}\mathbf{X}^\top\boldsymbol{\epsilon} = \frac{1}{n}\sum_{i=1}^{n} \epsilon^{(i)}\mathbf{x}^{(i)}.$$

Each coordinate of the estimation error averages $n$ independent terms
$\epsilon^{(i)} x^{(i)}_j$, each with mean $0$ and variance $\sigma^2$,
since the noise is independent of the features and $x_j^2$ has mean $1$.
Such an average has standard deviation $\sigma/\sqrt{n}$, which
that section calls its standard error. With $\sigma = 0.01$ and
$n = 1{,}000$ training examples, every coordinate of $\hat{\mathbf{w}}$,
the bias included, misses its true value by about
$0.01/\sqrt{1000} \approx 3 \times 10^{-4}$.

We check this prediction by solving the least-squares problem on the
training examples with `lstsq`, as in
that section.

In [5]:
n = data.num_train
X = jnp.concatenate((data.X[:n], jnp.ones((n, 1))), 1)  # Ones for b
w_hat = jnp.linalg.lstsq(X, data.y[:n])[0][:, 0]

We print the estimation errors of the two weights and of the bias, and the
standard error that we predicted for each.

In [6]:
e_w = d2l.numpy(w_hat[:2] - data.w)
print(f'error in w: ({e_w[0]:.1e}, {e_w[1]:.1e})')
print(f'error in b: {float(w_hat[2] - data.b):.1e}')
print(f'standard error: {data.noise / n ** 0.5:.1e}')

error in w: (-1.3e-04, 1.0e-04)
error in b: 1.5e-04
standard error: 3.2e-04


All three estimation errors are at most a few times $10^{-4}$, the size of
the standard error. For least squares, an error above $10^{-3}$, more than
three standard errors, occurs in fewer than one run in a hundred. This gives
the next section its threshold. Minibatch stochastic gradient descent on the
squared loss approaches the same least-squares estimate, but its random
minibatches keep it fluctuating around that estimate by a similar amount
(the noise ball of that section). A correct
implementation, trained until the loss has converged, therefore lands within
about $10^{-3}$ of the true parameters, and essentially always within
$2 \times 10^{-3}$. An error of $0.1$ cannot be blamed on the noise.
It points to a bug or to an optimization that has not converged; once the
loss has converged, to a bug. The threshold depends on the data: it shrinks
with more examples and less noise (exercise
5).

## Reading the Data in Minibatches

Minibatch stochastic gradient descent
(that section) updates the parameters
with the gradient averaged over a minibatch, a small set of examples drawn
at random. Because the minibatch is random, its average gradient is an
unbiased estimate of the gradient of the full loss; minibatches taken in a
fixed order, for instance sorted by label, would bias every step. Training
makes many passes over the data, called epochs. The usual way to draw the
minibatches is to shuffle the training examples at the start of every epoch
and to cut the shuffled sequence into consecutive pieces of `batch_size`
examples. Every example is then used once
per epoch, and the minibatches differ from one epoch to the next. The
validation examples need no shuffling, since we only average the loss over
them. We read them in their stored order, which makes runs easier to compare
when debugging.

We implement this procedure as the method `get_dataloader(train)`, which
`train_dataloader` and `val_dataloader` call with `train=True` and
`train=False`. Since `SyntheticRegressionData` is already defined, we
attach the method with the decorator `add_to_class` of
that section, which adds a function to an existing class as
a method. Because its body contains `yield`, calling the method returns a
generator, an object that runs the body piece by piece as a loop iterates
over it: each `yield` hands the loop one minibatch, a pair of features and
labels.

In [7]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    n = self.num_train
    if train:  # A fresh random order, drawn by NumPy on the host
        indices = np.random.default_rng().permutation(n)
    else:      # The validation examples, in a fixed order
        indices = np.arange(n, n + self.num_val)
    for i in range(0, len(indices), self.batch_size):
        batch_indices = indices[i : i + self.batch_size]
        yield self.X[batch_indices], self.y[batch_indices]

The first minibatch holds the features of 32 examples, two for each, and
their 32 labels.

In [8]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: (32, 2) 
y shape: (32, 1)


With 1,000 training examples and minibatches of 32, an epoch consists of
$\lceil 1000/32 \rceil = 32$ minibatches: 31 full ones and a last one with
the remaining 8 examples. We count the minibatches of one pass over the
training loader, and then try a second pass over the same loader.

In [9]:
loader = data.train_dataloader()
sizes = [len(y) for X, y in loader]
print(f'{len(sizes)} minibatches, the last of size {sizes[-1]}')
print(f'second pass: {len([y for X, y in loader])} minibatches')

32 minibatches, the last of size 8
second pass: 0 minibatches


The count matches, but the second pass yields nothing. A generator can be
iterated only once; after one pass it is exhausted. This matters for
training. The `Trainer` of that section asks the `DataModule`
for its loaders once and then iterates over them in every epoch, so with
our generator it would train for one epoch and then do nothing, without an
error. Our loader has further limits. It cannot report how many minibatches
an epoch has without running through it, it holds all the data in memory,
and it assembles every minibatch in the same Python process that trains the
model.

## Reusing a Loader Across Epochs

A reusable data loader removes the first two limits. It can be iterated in
every epoch, draws a fresh random order on every pass, and reports its
number of minibatches through `len`. The loaders that deep learning
frameworks provide also remove the other two. For data that do not fit in
memory, or that must be decoded and transformed before use, such as large
collections of images, they read examples from disk and prepare the next
minibatches in parallel workers while the model trains on the current one.
Our dataset fits in memory, so we need only the first two properties.

For data that are already in memory as tensors, we add the method
`get_tensorloader` to the base class `DataModule`, so that every such
dataset can use it. It takes a tuple of tensors whose first axis indexes the
examples, keeps the examples selected by `indices`, and returns a loader
that shuffles them if `train` is true. Where the framework provides a
loader that suits such data, `get_tensorloader` returns it; otherwise we
write a short class of our own.

`ArrayLoader` keeps the arrays on the host as NumPy arrays and yields each
minibatch as NumPy arrays, which JAX functions accept and move to the
device. When it shuffles, it splits a new key off its own key on every pass
and draws the order from that key, so that every pass has a new order and
no key is used twice. It keeps the last, partial minibatch unless we ask it
to drop it. The compiled training
and validation steps, which we write in that section, are
compiled once for every new input shape, so the last minibatch costs one
more compilation of each step per run, not one per epoch.

In [10]:
class ArrayLoader:
    """Minibatches of in-memory arrays, reshuffled each pass if shuffle."""
    def __init__(self, arrays, batch_size, shuffle=False, drop_last=False,
                 key=None):
        self.arrays = tuple(np.asarray(a) for a in arrays)  # On the host
        self.batch_size, self.shuffle = batch_size, shuffle
        self.drop_last = drop_last
        if shuffle and key is None:  # A new seed on each run
            key = jax.random.key(random.getrandbits(32))
        self.key = key

    def __len__(self):
        n = len(self.arrays[0])
        if self.drop_last:
            return n // self.batch_size
        return (n + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        n = len(self.arrays[0])
        if self.shuffle:  # A new key, and hence a new order, on every pass
            self.key, key = jax.random.split(self.key)
            order = np.asarray(jax.random.permutation(key, n))
        else:
            order = np.arange(n)
        for i in range(len(self)):
            start = i * self.batch_size
            batch_indices = order[start : start + self.batch_size]
            yield tuple(a[batch_indices] for a in self.arrays)

`get_tensorloader` wraps the selected arrays in an `ArrayLoader` that
shuffles only for training.

In [11]:
@d2l.add_to_class(d2l.DataModule)
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    return ArrayLoader(tensors, self.batch_size, shuffle=train)

`SyntheticRegressionData` now serves its two splits through
`get_tensorloader`. The new `get_dataloader` replaces the generator above:
the training split holds the first `num_train` examples, and the
validation split the rest.

In [12]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

We repeat the test of the generator: one pass over the training loader,
then a second pass over the same loader.

In [13]:
loader = data.train_dataloader()
sizes = [len(y) for X, y in loader]
print(f'{len(sizes)} minibatches, the last of size {sizes[-1]}')
print(f'second pass: {len([y for X, y in loader])} minibatches')
for _ in range(2):  # Each new pass starts in a new random order
    X, y = next(iter(loader))
    print('a new pass starts with labels', d2l.numpy(y[:3, 0]).round(2))

32 minibatches, the last of size 8
second pass: 32 minibatches
a new pass starts with labels [ 2.26  8.6  -1.48]
a new pass starts with labels [-3.16 14.05  0.4 ]


Like our generator, the new loader yields 32 minibatches, and the last one
holds 8 examples. Unlike the generator, it yields all 32 again on the second
pass, and every pass starts with different examples because the order is
drawn anew. It also reports the number of minibatches without a pass.

In [14]:
len(loader)

32

From now on, every `DataModule` that holds its data as tensors serves them
through `get_tensorloader`.

We now have a dataset whose true parameters we know, the size of the error
that its noise alone causes, about $3 \times 10^{-4}$, and a loader that
serves the training examples in a new random order every epoch. The next
section implements the linear model, the squared loss, and minibatch
stochastic gradient descent, trains them on this dataset, and compares the
estimates with $\mathbf{w}^*$ and $b^*$. Passing this test tells us that the
implementation recovers parameters that it should recover. It does not tell
us how the method behaves when the model does not match the data, which is
the usual case with real data.

## Summary

A poor result on real data cannot tell a bug from a wrong model; a
synthetic dataset can. We generated labels from a linear function with
known parameters $\mathbf{w}^* = (2, -3.4)$ and $b^* = 4.2$ plus Gaussian
noise, for which a linear model with the squared loss is correctly
specified. The noise moves even the exact least-squares estimate away from
the truth, by about the standard error
$\sigma/\sqrt{n} \approx 3 \times 10^{-4}$, so a correct implementation
trained to convergence should land within a threshold of about $10^{-3}$.
`SyntheticRegressionData` packages the generator as a `DataModule`. Its
loader shuffles the training examples at the start of every epoch and cuts
them into minibatches, 32 per epoch with the last holding 8 examples. A
hand-written generator shows the mechanism but can be iterated only once,
so we use a loader that can be iterated in every epoch and reports its
length: the framework's own, or a short class of our own.


## Exercises

1. [code] **Reproducibility.** `SyntheticRegressionData` splits its `key` into
   `key1` for `X` and `key2` for `eps`.
    1. Explain why two instances built with the same `key` hold identical
       data without any global seed, and why two instances built without a
       `key` do not.
    1. Change both draws to use `key` itself instead of `key1` and `key2`,
       and compare `eps / noise` with the entries of `X` numerically. State
       what the split guarantees about `X` and `eps` that this version does
       not.
    1. Construct five instances whose datasets are independent of each
       other, as averaging an error over repeated trials requires, and state
       how you derive their keys.

2. **Planted bug.** A faulty variant of the hand-written `get_dataloader`
   shuffles the rows of `X` and the entries of `y` with two independent
   permutations. We collect the pairs that it yields in one training epoch
   on `data`, whose parameters are $\mathbf{w}^* = (2, -3.4)$ and
   $b^* = 4.2$ with noise standard deviation $0.01$. We then fit
   $\hat{\mathbf{w}}$ and $\hat{b}$ to these pairs by least squares.
    1. Predict the approximate values of $\hat{\mathbf{w}}$ and $\hat{b}$,
       and explain your prediction.
    1. Estimate the mean squared error of this fit on the collected pairs,
       and compare it with the smallest mean squared error that any
       predictor can attain on correctly paired data.

3. **A partial last minibatch.** With 1,000 examples per split and
   `batch_size=32`, each loader ends a pass with a partial minibatch of 8
   examples. `ArrayLoader` discards it when given `drop_last=True`.
    1. The training loader reshuffles every epoch, whereas the validation
       loader reads in a fixed order. For each loader, state which examples
       a discarded partial minibatch leaves out over many epochs, and decide
       for which loader discarding is acceptable.
    1. A compiled step is compiled once for every new input shape. Count
       the compilations of the training and validation steps that keeping
       the partial minibatch causes over ten epochs, and compare their cost
       with that of discarding 8 examples in every epoch.

4. [code] **A loader for every epoch.** `ArrayLoader` and the generator of
   this section cut the examples into minibatches in the same way, but only
   `ArrayLoader` can be iterated in every epoch.
    1. Python distinguishes an *iterable*, whose `__iter__` returns a new
       iterator on every call, from an *iterator*, whose `__iter__` returns
       the iterator itself and which is exhausted after one pass. Say
       which of the two the generator is and which `ArrayLoader` is, and
       how this explains the empty second pass.
    1. Check `len` against the number of minibatches that a pass yields,
       with and without `drop_last`.
    1. Check that two passes each visit every training example exactly
       once, and that their orders differ. Then build two loaders with the
       same `key` and show that they produce the same sequence of orders.
       Explain what would go wrong if `__iter__` drew every order from
       `self.key` without splitting it.

5. [code] **Recovery under noise.** ● Repeat the least-squares fit of
   that section on 1,000 independent
   datasets for each setting below, and measure the standard deviation of
   the estimation error $\hat{w}_1 - w_1^*$ of the first weight.
    1. Vary `noise` over $\{0.001, 0.01, 0.1, 1\}$ with `num_train=1000`,
       and `num_train` over $\{10, 100, 1{,}000, 10{,}000\}$ with
       `noise=0.01`. Plot both measurements against the prediction
       $\sigma/\sqrt{n}$ on logarithmic axes.
    1. Find the smallest `num_train` at which the measurement agrees with
       $\sigma/\sqrt{n}$ to within 10%, and say which step of the derivation
       fails for fewer examples.
    1. Derive $\textrm{Cov}(\hat{\mathbf{w}}) = \sigma^2(\mathbf{X}^\top\mathbf{X})^{-1}$
       from the expression for $\hat{\mathbf{w}} - \mathbf{w}^*$ in this
       section. Then generate correlated features
       $x_2 = \rho x_1 + \sqrt{1 - \rho^2}\, z$, with $x_1$ and $z$
       independent and standard normal, for $\rho \in \{0, 0.9, 0.99\}$.
       Predict the standard error of $\hat{w}_1$ from this formula, check
       it numerically, and explain why the predictions
       $\hat{\mathbf{w}}^\top \mathbf{x}$ on new examples drawn from the same
       distribution remain accurate when the weights do not.

6. [code] **On-the-fly generation.** Write a `DataModule` subclass that
   stores no $\mathbf{X}$ or $\mathbf{y}$. Its training loader instead draws
   every minibatch fresh from the data-generating process when the
   minibatch is requested.
    1. A stream of fresh minibatches has no natural end, yet the `Trainer`
       ends an epoch only when a pass over the training loader ends. Decide
       how many minibatches one epoch yields, make every pass stop there,
       and make `len` return that number.
    1. Decide whether the validation loader should also draw fresh examples
       on every pass, and implement your choice. Justify it by what the
       validation loss is used for.

7. **Data beyond memory.** ● Suppose that `len(w)` and
   `num_train + num_val` are so large that $\mathbf{X}$ no longer fits in
   memory, so `__init__` generates it in chunks and writes it to disk.
    1. Name the statements in `SyntheticRegressionData.__init__` and in the
       hand-written `get_dataloader` whose memory or cost grows with the
       number of examples, and state which of them writing $\mathbf{X}$ to
       disk in chunks does not fix.
    1. Reading single examples from random disk locations is slow, so the
       training loader should read long contiguous blocks. Design a loader
       that reads such blocks and still visits the examples in a fresh order
       every epoch. State in what sense this order falls short of a
       uniformly random permutation.
    1. The loader must also avoid storing a permutation table with one
       entry per example. A *pseudorandom permutation*
       [@Naor.Reingold.1999] is a bijection of $\{0, \ldots, n-1\}$,
       selected by a key, that computes the position of any index without a
       table, for instance with a few rounds of a Feistel network. Explain
       how it meets this requirement while still changing the order every
       epoch.

[Discussions](https://d2l.discourse.group/t/17975)